# 🎮 Caso 02: PlayLoop — Mobile Gaming Analytics
## 📅 Día 1: Ingesta, Auditoría Forense y Modelo de Datos (Star Schema)

**Estudio:** PlayLoop Studios (Barcelona)  
**Modelo de Negocio:** Free-to-Play (F2P) con monetización híbrida (IAP + Ads)  

---

### 👥 Los Stakeholders y sus preguntas:
- **Diego (Senior Data Mentor):** *"En mobile no compras clientes, compras comportamiento. Antes de calcular LTV o retención, audita si las 7 tablas tienen integridad referencial o si hay jugadores fantasma."*
- **Laura (Head of Product):** *"¿La granularidad de sesiones y partidas cuadra con los usuarios activos?"*
- **Carlos (UA / Marketing Lead):** *"¿Las instalaciones registradas en el juego coinciden con las que facturan las redes de adquisición?"*

### 1. Importación de Librerías y Configuración

In [1]:
# 1. Pandas: Manipulación y análisis de estructuras tabulares (DataFrames)
import pandas as pd

# 2. NumPy: Operaciones numéricas vectorizadas de alto rendimiento
import numpy as np

# 3. Matplotlib (Pyplot): Motor gráfico base para dibujar gráficos y figuras
import matplotlib.pyplot as plt

# 4. Seaborn: Capa estadística sobre Matplotlib para visualizaciones estéticas
import seaborn as sns

# 5. OS: Módulo nativo del sistema operativo para gestionar rutas de archivos de forma agnóstica a Windows/Linux
import os

# 6. RE: Módulo de expresiones regulares para limpieza avanzada de textos y patrones
import re

# Configuración 1: Obliga a Pandas a mostrar todas las columnas sin truncarlas con puntos suspensivos (...)
pd.set_option('display.max_columns', None)

# Configuración 2: Formatea números decimales con separador de miles y dos decimales (ej. 1,250.50 en vez de 1250.50123)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

# Configuración 3: Aplica una plantilla visual limpia de fondo blanco con cuadrícula suave para los gráficos
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

# Variable constante: Ruta relativa donde residen los archivos CSV originales descargados
DATA_DIR = 'data/raw'

# Mensaje de validación para confirmar que todas las librerías se cargaron en memoria
print('✅ Entorno y librerías configuradas correctamente.')

✅ Entorno y librerías configuradas correctamente.


### 2. Carga de los 7 Datasets del Data Warehouse

In [2]:
# --- TABLAS DE DIMENSIÓN (Quién y Qué) ---
# os.path.join une 'data/raw' y 'usuarios.csv' con la barra correcta según el SO (Windows: data\raw\usuarios.csv)
# pd.read_csv lee el archivo CSV y lo carga en memoria RAM estructurado como tabla bidimensional (DataFrame)
df_usuarios = pd.read_csv(os.path.join(DATA_DIR, 'usuarios.csv'))
df_productos = pd.read_csv(os.path.join(DATA_DIR, 'productos_iap.csv'))

# --- TABLAS DE HECHOS DE COMPORTAMIENTO (Eventos de actividad y gameplay) ---
# Registra cada apertura de la app, duración de la sesión en segundos y dispositivo
df_sesiones = pd.read_csv(os.path.join(DATA_DIR, 'sesiones.csv'))
# Registra cada partida jugada, nivel, si ganó/perdió, movimientos y boosters utilizados
df_partidas = pd.read_csv(os.path.join(DATA_DIR, 'partidas.csv'))

# --- TABLAS DE HECHOS DE MONETIZACIÓN (Ingresos directos e indirectos) ---
# Registra transacciones con dinero real (compras in-app dentro de la tienda del juego)
df_compras = pd.read_csv(os.path.join(DATA_DIR, 'compras.csv'))
# Registra las impresiones de anuncios publicitarios vistos por los jugadores e ingresos generados
df_ads = pd.read_csv(os.path.join(DATA_DIR, 'ingresos_ads.csv'))

# --- TABLA DE HECHOS DE ADQUISICIÓN (Costes de Marketing) ---
# Registra el gasto diario por canal publicitario (Google, FB, TikTok, etc.), impresiones y clics
df_gasto_ua = pd.read_csv(os.path.join(DATA_DIR, 'gasto_ua.csv'))

# --- CONSTRUCCIÓN DEL RESUMEN ESTRUCTURAL ---
# Diccionario clave:valor donde agrupamos los 7 DataFrames con una etiqueta descriptiva de su rol arquitectónico
tablas = {
    'usuarios (Dim)': df_usuarios,
    'productos_iap (Dim)': df_productos,
    'sesiones (Hecho)': df_sesiones,
    'partidas (Hecho)': df_partidas,
    'compras_iap (Hecho)': df_compras,
    'ingresos_ads (Hecho)': df_ads,
    'gasto_ua (Hecho/Agregado)': df_gasto_ua
}

# List Comprehension: Itera sobre cada par (nombre, df). Para cada tabla, extrae:
# df.shape[0] -> cantidad total de filas (registros/eventos)
# df.shape[1] -> cantidad total de columnas (atributos/variables)
# pd.DataFrame(...) -> Convierte la lista de diccionarios resultante en una tabla visual limpia
resumen = pd.DataFrame([{'Tabla': nombre, 'Filas': df.shape[0], 'Columnas': df.shape[1]} for nombre, df in tablas.items()])

# Mostrar la tabla resumen en el notebook
resumen

,Tabla,Filas,Columnas
0,usuarios (Dim),8040,6
1,productos_iap (Dim),14,5
2,sesiones (Hecho),44945,5
3,partidas (Hecho),61716,7
4,compras_iap (Hecho),4558,8
5,ingresos_ads (Hecho),19999,6
6,gasto_ua (Hecho/Agregado),280,6


### 3. Auditoría Forense: Detección de Valores Nulos y Formatos Sucios

In [3]:
# --- AUDITORÍA DE NULOS EN CADA TABLA ---
# Recorremos las 7 tablas buscando si existen campos vacíos o incompletos
print('🔍 REPORTE FORENSE DE VALORES NULOS:')
for nombre, df in tablas.items():
    # df.isnull() evalúa celda por celda; .sum() cuenta cuántos True (nulos) hay en cada columna
    nulos = df.isnull().sum()
    # Filtramos sólo las columnas que tengan al menos 1 valor nulo
    nulos_detectados = nulos[nulos > 0]
    if not nulos_detectados.empty:
        print(f'\n⚠️ {nombre}:')
        for col, cant in nulos_detectados.items():
            pct = (cant / len(df)) * 100
            print(f'   - Columna "{col}": {cant:,} nulos ({pct:.2f}% de la tabla)')

print('\n' + '='*60 + '\n')

# --- AUDITORÍA DE FORMATOS SUCIOS EN DINERO (Strings en lugar de números) ---
# Comprobamos los tipos de datos de las columnas monetarias y mostramos muestras reales
print('🔍 MUESTRA DE FORMATOS MONETARIOS ORIGINALES:')
print(f'df_compras["importe"] (Tipo: {df_compras["importe"].dtype}):', df_compras['importe'].head(3).tolist())
print(f'df_gasto_ua["gasto"] (Tipo: {df_gasto_ua["gasto"].dtype}):', df_gasto_ua['gasto'].head(3).tolist())
print(f'df_productos["precio_base"] (Tipo: {df_productos["precio_base"].dtype}):', df_productos['precio_base'].head(3).tolist())

🔍 REPORTE FORENSE DE VALORES NULOS:

⚠️ usuarios (Dim):
   - Columna "version_app": 297 nulos (3.69% de la tabla)

⚠️ productos_iap (Dim):
   - Columna "monedas": 3 nulos (21.43% de la tabla)

⚠️ partidas (Hecho):
   - Columna "boosters_usados": 1,854 nulos (3.00% de la tabla)

⚠️ ingresos_ads (Hecho):
   - Columna "ingreso": 600 nulos (3.00% de la tabla)

⚠️ gasto_ua (Hecho/Agregado):
   - Columna "clics": 11 nulos (3.93% de la tabla)


🔍 MUESTRA DE FORMATOS MONETARIOS ORIGINALES:
df_compras["importe"] (Tipo: str): ['4,99 €', '0,99 €', '0,99 €']
df_gasto_ua["gasto"] (Tipo: str): ['11,53 €', '20,60 €', '0,00 €']
df_productos["precio_base"] (Tipo: str): ['0,99 €', '2,99 €', '4,99 €']


### 4. Limpieza y Normalización: Fechas, Nulos e Importes Multidivisa (EUR vs MXN)

In [4]:
# 1. Conversión de fechas multiformato a datetime64 (ISO, barras DD/MM/AAAA y texto en español)
df_usuarios['fecha_instalacion'] = pd.to_datetime(df_usuarios['fecha_instalacion'], format='mixed', dayfirst=True)
df_sesiones['fecha'] = pd.to_datetime(df_sesiones['fecha'], format='mixed', dayfirst=True)
df_partidas['fecha'] = pd.to_datetime(df_partidas['fecha'], format='mixed', dayfirst=True)
df_compras['fecha'] = pd.to_datetime(df_compras['fecha'], format='mixed', dayfirst=True)
df_ads['fecha'] = pd.to_datetime(df_ads['fecha'], format='mixed', dayfirst=True)
df_gasto_ua['fecha'] = pd.to_datetime(df_gasto_ua['fecha'], format='mixed', dayfirst=True)

# 2. Imputación de nulos según lógica de negocio
# Si no hay registro de boosters usados en una partida, significa que se usaron 0 boosters
df_partidas['boosters_usados'] = df_partidas['boosters_usados'].fillna(0)
# Si un producto no tiene monedas asociadas en el catálogo (ej. Quitar Anuncios), son 0 monedas
df_productos['monedas'] = df_productos['monedas'].fillna(0)
# Si un anuncio tuvo un fallo o no generó ingreso, se imputa como 0.0 €
df_ads['ingreso'] = df_ads['ingreso'].fillna(0.0)
# Si no se registraron clics en un día de campaña, se imputa como 0
df_gasto_ua['clics'] = df_gasto_ua['clics'].fillna(0)

# 3. Función de limpieza de importes en compras (gestión de multidivisa MXN y reembolsos negativos)
def limpiar_importe_compras(row):
    val = str(row['importe'])
    es_mxn = ('MXN' in val) or (row['moneda'] == 'MXN')
    # 1. Quita puntos de miles si existieran (ej. '1.000,50' -> '1000,50')
    val = val.replace('.', '')
    # 2. Convierte coma decimal a punto (ej. '1000,50' -> '1000.50')
    val = val.replace(',', '.')
    # 3. Expresión regular: conserva solo dígitos, el punto decimal y el signo menos de reembolsos
    val_limpio = re.sub(r'[^\d.-]', '', val)
    num = float(val_limpio)
    # Si es MXN, convertimos a EUR usando la tasa de paridad de catálogo (19.50 MXN = 1 EUR)
    if es_mxn:
        return round(num / 19.50, 2)
    return num

# Aplicamos la función a cada fila de df_compras
df_compras['importe_eur'] = df_compras.apply(limpiar_importe_compras, axis=1)

# 4. Función vectorizada robusta para columnas en euros con separadores de miles europeos (1.263,71 €)
def limpiar_euros_europeos(serie):
    return (
        serie.astype(str)
        .str.replace('.', '', regex=False)        # 1. Elimina el punto de miles ('1.263,71 €' -> '1263,71 €')
        .str.replace(',', '.', regex=False)        # 2. Cambia coma decimal por punto ('1263,71 €' -> '1263.71 €')
        .str.replace(r'[^\d.-]', '', regex=True)   # 3. Elimina el símbolo de euro y espacios ('1263.71 €' -> '1263.71')
        .astype(float)                             # 4. Convierte de forma segura a float numérico
    )

# Limpieza de df_gasto_ua y df_productos_iap
df_gasto_ua['gasto_eur'] = limpiar_euros_europeos(df_gasto_ua['gasto'])
df_productos['precio_eur'] = limpiar_euros_europeos(df_productos['precio_base'])

print('✅ Limpieza completada exitosamente.')
print(f'Total recaudado en compras IAP: {df_compras["importe_eur"].sum():,.2f} €')
print(f'Total invertido en marketing UA: {df_gasto_ua["gasto_eur"].sum():,.2f} €')

✅ Limpieza completada exitosamente.
Total recaudado en compras IAP: 10,888.58 €
Total invertido en marketing UA: 9,060.01 €


### 5. Integridad Referencial de Usuarios y Deduplicación de la Llave Primaria

In [5]:
# 1. Detección de duplicados en la dimensión de Usuarios
# Evaluamos cuántas filas comparten el mismo user_id
duplicados_usuarios = df_usuarios[df_usuarios.duplicated(subset=['user_id'], keep=False)]
print(f'⚠️ Filas duplicadas encontradas en df_usuarios: {len(duplicados_usuarios)} filas ({len(duplicados_usuarios)//2} IDs)')

# 2. Deduplicación conservando el registro más actualizado
# En un Data Warehouse, la dimensión debe tener llaves primarias únicas para evitar multiplicar transacciones en los JOINs
df_usuarios = df_usuarios.drop_duplicates(subset=['user_id'], keep='last').copy()
print(f'✅ Dimensión deduplicada: {df_usuarios.shape[0]:,} usuarios únicos garantizados.')

# 3. Normalización de plataformas (iOS vs Android)
# Estandarizamos mayúsculas y minúsculas erráticas (ios, iOS, IOS -> iOS; android, Android -> Android)
df_usuarios['plataforma'] = df_usuarios['plataforma'].str.strip().str.capitalize()
df_usuarios['plataforma'] = df_usuarios['plataforma'].replace({'Ios': 'iOS', 'Android': 'Android'})

# 4. Auditoría de Integridad Referencial (¿Existen eventos de jugadores fantasma?)
# Extraemos el conjunto de IDs válidos de la dimensión
usuarios_validos = set(df_usuarios['user_id'])

# Calculamos la diferencia de conjuntos (IDs en hechos que NO existen en dimensiones)
huerfanos_sesiones = len(set(df_sesiones['user_id']) - usuarios_validos)
huerfanos_partidas = len(set(df_partidas['user_id']) - usuarios_validos)
huerfanos_compras = len(set(df_compras['user_id']) - usuarios_validos)
huerfanos_ads = len(set(df_ads['user_id']) - usuarios_validos)

print('\n🔍 REPORTE DE INTEGRIDAD REFERENCIAL (Eventos Huérfanos):')
print(f'   - Sesiones huérfanas: {huerfanos_sesiones:,}')
print(f'   - Partidas huérfanas: {huerfanos_partidas:,}')
print(f'   - Compras huérfanas: {huerfanos_compras:,}')
print(f'   - Anuncios huérfanos: {huerfanos_ads:,}')
print('✅ Integridad referencial perfecta: 100% de los eventos corresponden a usuarios reales.')

⚠️ Filas duplicadas encontradas en df_usuarios: 80 filas (40 IDs)
✅ Dimensión deduplicada: 8,000 usuarios únicos garantizados.

🔍 REPORTE DE INTEGRIDAD REFERENCIAL (Eventos Huérfanos):
   - Sesiones huérfanas: 0
   - Partidas huérfanas: 0
   - Compras huérfanas: 0
   - Anuncios huérfanos: 0
✅ Integridad referencial perfecta: 100% de los eventos corresponden a usuarios reales.


### 6. Normalización de Canales de Adquisición y Reconciliación con Marketing (UA)

In [6]:
# 1. Función de mapeo y normalización de canales publicitarios
# Reduce 23 etiquetas caóticas (mayúsculas, guiones, typos) a los 6 canales reales del negocio
def normalizar_canal(c):
    c_clean = str(c).lower().strip().replace('_', ' ').replace('-', ' ')
    if 'apple' in c_clean or 'asa' in c_clean:
        return 'Apple Search Ads'
    elif 'meta' in c_clean or 'facebook' in c_clean:
        return 'Meta Ads'
    elif 'google' in c_clean:
        return 'Google Ads'
    elif 'tik' in c_clean:
        return 'TikTok'
    elif 'coin' in c_clean:
        return 'CoinRush'
    elif 'org' in c_clean:
        return 'Orgánico'
    return 'Otro'

# Aplicamos la normalización a ambas tablas
df_usuarios['canal_norm'] = df_usuarios['canal_adquisicion'].apply(normalizar_canal)
df_gasto_ua['canal_norm'] = df_gasto_ua['canal'].apply(normalizar_canal)

# 2. Agrupación y cálculo de instalaciones por canal
# Usuarios registrados reales en la app por canal
usuarios_por_canal = df_usuarios['canal_norm'].value_counts().rename('Usuarios_Registrados')
# Instalaciones facturadas por las redes de marketing en gasto_ua
instalaciones_ua = df_gasto_ua.groupby('canal_norm')['instalaciones'].sum().rename('Instalaciones_Facturadas')

# 3. Tabla de Reconciliación Forense
tabla_reconciliacion = pd.concat([usuarios_por_canal, instalaciones_ua], axis=1)
tabla_reconciliacion['Diferencia_Absoluta'] = tabla_reconciliacion['Instalaciones_Facturadas'] - tabla_reconciliacion['Usuarios_Registrados']
tabla_reconciliacion['Discrepancia_Pct'] = (tabla_reconciliacion['Diferencia_Absoluta'] / tabla_reconciliacion['Usuarios_Registrados']) * 100

# Ordenamos por volumen de usuarios
tabla_reconciliacion = tabla_reconciliacion.sort_values('Usuarios_Registrados', ascending=False)
tabla_reconciliacion

,Usuarios_Registrados,Instalaciones_Facturadas,Diferencia_Absoluta,Discrepancia_Pct
canal_norm,,,,
CoinRush,2800,2800,0,0.00
Meta Ads,1600,1600,0,0.00
Google Ads,1400,1400,0,0.00
Apple Search Ads,900,900,0,0.00
TikTok,700,700,0,0.00
Orgánico,600,600,0,0.00


### 7. Normalización de Eventos y Exportación del Data Warehouse Limpio

In [7]:
# 1. Normalización categórica de partidas (ganadas vs perdidas)
mapeo_partidas = {
    'win': 'Victoria', 'WIN': 'Victoria', 'ganada': 'Victoria',
    'fail': 'Derrota', 'lose': 'Derrota', 'perdida': 'Derrota'
}
df_partidas['resultado'] = df_partidas['resultado'].str.strip().replace(mapeo_partidas)

# 2. Normalización de formatos y redes de anuncios publicitarios
mapeo_tipo_ad = {
    'interstitial': 'Interstitial', 'Interstitial': 'Interstitial',
    'rewarded': 'Rewarded', 'Rewarded': 'Rewarded', 'recompensado': 'Rewarded',
    'banner': 'Banner', 'Banner': 'Banner'
}
df_ads['tipo_ad'] = df_ads['tipo_ad'].str.strip().replace(mapeo_tipo_ad)

mapeo_red_ad = {
    'admob': 'AdMob', 'AdMob': 'AdMob',
    'ironsource': 'IronSource', 'ironSource': 'IronSource'
}
df_ads['red_ad'] = df_ads['red_ad'].str.strip().replace(mapeo_red_ad)

# 3. Exportación de las 7 tablas limpias y estandarizadas a data/clean/
OUTPUT_DIR = 'data/clean'
os.makedirs(OUTPUT_DIR, exist_ok=True)

df_usuarios.to_csv(os.path.join(OUTPUT_DIR, 'usuarios_clean.csv'), index=False)
df_sesiones.to_csv(os.path.join(OUTPUT_DIR, 'sesiones_clean.csv'), index=False)
df_partidas.to_csv(os.path.join(OUTPUT_DIR, 'partidas_clean.csv'), index=False)
df_compras.to_csv(os.path.join(OUTPUT_DIR, 'compras_clean.csv'), index=False)
df_ads.to_csv(os.path.join(OUTPUT_DIR, 'ingresos_ads_clean.csv'), index=False)
df_gasto_ua.to_csv(os.path.join(OUTPUT_DIR, 'gasto_ua_clean.csv'), index=False)
df_productos.to_csv(os.path.join(OUTPUT_DIR, 'productos_iap_clean.csv'), index=False)

print('✅ 7 Datasets normalizados y exportados exitosamente a data/clean/')
print('Archivos generados en data/clean/:')
for f in sorted(os.listdir(OUTPUT_DIR)):
    tam_kb = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1024
    print(f'   - {f} ({tam_kb:,.1f} KB)')

✅ 7 Datasets normalizados y exportados exitosamente a data/clean/
Archivos generados en data/clean/:
   - compras_clean.csv (250.9 KB)
   - embudo_niveles_gameplay.csv (1.3 KB)
   - gasto_ua_clean.csv (16.7 KB)
   - importancia_variables_churn.csv (0.6 KB)
   - ingresos_ads_clean.csv (990.4 KB)
   - kpis_engagement_mensual.csv (0.3 KB)
   - monetizacion_usuarios.csv (531.1 KB)
   - partidas_clean.csv (2,622.8 KB)
   - predicciones_churn_usuarios.csv (1,089.2 KB)
   - productos_iap_clean.csv (0.7 KB)
   - ranking_canales_retencion.csv (0.4 KB)
   - rentabilidad_canales_roas.csv (0.8 KB)
   - retencion_curva_global.csv (0.4 KB)
   - sesiones_clean.csv (1,559.0 KB)
   - usuarios_clean.csv (403.7 KB)


### 8. 📋 Resumen Ejecutivo y Respuestas a Stakeholders (Día 1)

| Stakeholder | Pregunta Planteada | Hallazgo Forense | Conclusión de Negocio |
| :--- | :--- | :--- | :--- |
| **Diego (Senior Mentor)** | *¿Hay jugadores fantasma o fallos de integridad referencial?* | **0 eventos huérfanos**. La dimensión `usuarios` tenía 40 duplicados (80 filas) que se limpiaron. | El tracking de eventos es 100% confiable. La base cuenta con **8,000 usuarios únicos exactos**. |
| **Laura (Head of Product)** | *¿La granularidad de sesiones y partidas cuadra con los usuarios activos?* | Se mapearon 44,945 sesiones y 61,716 partidas. Había inconsistencia de idiomas en resultados (`win`, `ganada`, `lose`). | Normalizado a `Victoria` y `Derrota`. Todas las sesiones pertenecen a jugadores reales. |
| **Carlos (UA / Marketing)** | *¿Las instalaciones registradas coinciden con lo que cobran las redes publicitarias?* | Aparentaba discrepancias graves por 23 variantes tipográficas (`ASA`, `meta`, `coinrush`). | **0.00% de discrepancia**. Al unificar en los 6 canales oficiales, los 8,000 usuarios coinciden exactamente con las 8,000 descargas facturadas. |